# YOLO 3회차 산업 데모 ① — 회전 박스(OBB)로 항공 사진 속 차량·선박 찾기

항만, 주차장, 물류 야적장처럼 **위에서 내려다본 사진**에서는 차량·선박·컨테이너가 비스듬히 놓여 있습니다. 지금까지 쓴 박스(가로·세로가 화면과 나란한 직사각형)로 감싸면 물체 사이의 빈 땅까지 박스에 들어가고, 나란히 붙은 물체끼리 박스가 크게 겹칩니다.
**회전 박스(OBB, Oriented Bounding Box — 물체 방향을 따라 기울어진 박스)** 는 각도까지 함께 예측해 이 문제를 줄입니다.

| 단계 | 하는 일 | 결과물 |
|---|---|---|
| Step 1 | 일반 박스와 회전 박스의 차이를 그림으로 보기 | 면적 비교 그림 |
| Step 2 | 사전학습 OBB 모델로 항공 사진 탐지 | `obb_preds.csv` |
| Step 3 | 나란히 붙은 물체의 겹침 비교 (일반 박스 vs 회전 박스) | 겹침 비교 표 |
| Step 4 | 규칙 — 마리나에서 정박 방향을 벗어난 선박 찾기 | `obb_marina.png` |
| Step 5 | (선택) 짧은 미세조정 | 학습 그래프 |

**데이터**: Ultralytics 공개 샘플 `DOTA8`(항공 사진 데이터셋 DOTA에서 8장만 뽑은 맛보기용, 15개 클래스)
**실행 환경**: Google Colab, 런타임 유형 **T4 GPU**(Colab에서 빌려 쓰는 NVIDIA 그래픽카드) — CPU에서도 1~2분이면 끝납니다.

## 0. 준비

In [ ]:
# 패키지 설치 — ultralytics(YOLO). -q: 설치 로그 줄임
!pip -q install ultralytics

In [ ]:
import os, json, shutil, base64, subprocess, time
from pathlib import Path

# 영상·이미지 처리(cv2), 배열(numpy), 표(pandas), 그래프(matplotlib)
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display  # 셀 출력에 HTML(영상 플레이어 등) 띄우기
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())  # T4 런타임이면 True

try:
    from google.colab import drive  # Colab에만 있는 모듈 → 성공하면 Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True  # False면 Drive 대신 런타임 임시 폴더에 저장 (연결이 끊기면 사라짐)
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day3/obb')  # Drive 안 결과 폴더
else:
    SAVE_DIR = Path('outputs/obb')
SAVE_DIR.mkdir(parents=True, exist_ok=True)  # 중간 폴더까지 생성, 이미 있으면 그냥 통과
print('결과 저장 위치:', SAVE_DIR.resolve())

In [ ]:
# OpenCV(BGR) 이미지를 matplotlib로 보여 주기
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')  # [..., ::-1]: 색 축을 뒤집어 BGR→RGB, 눈금 숨김
    if title:
        plt.title(title)
    plt.show()

# 여러 장을 격자로 보여 주기 (입력: BGR 이미지 목록, 제목 목록, 한 줄에 놓을 장 수)
def show_grid(imgs, titles=None, cols=3, size=4.5):
    rows = (len(imgs) + cols - 1) // cols  # 필요한 줄 수 (올림)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for k, im in enumerate(imgs):
        ax = axes.ravel()[k]
        ax.imshow(im[..., ::-1])
        if titles:
            ax.set_title(titles[k], fontsize=9)
    plt.tight_layout(); plt.show()

# mp4 영상을 노트북 안에서 재생 (입력: 영상 경로, 표시 폭 px)
# OpenCV가 만든 mp4v 영상은 브라우저가 못 틀어서 H.264로 다시 인코딩
def show_video(path, width=640):
    h264 = str(Path(path).with_name(Path(path).stem + '_h264.mp4'))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path),
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', h264], check=True)
    data = base64.b64encode(open(h264, 'rb').read()).decode()  # 영상을 글자(base64)로 바꿔 HTML에 직접 넣음
    display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

# 파일을 SAVE_DIR로 복사하고 복사된 경로를 돌려줌
def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():  # 원본과 대상이 같은 파일이면 복사 생략
        shutil.copy(path, dst)
    return dst

In [ ]:
# DOTA8 샘플 데이터 내려받기 (Ultralytics 공개 자산, GitHub)
DATA = Path('dota8')
if not DATA.exists():  # 이미 받았으면 건너뜀
    !wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/dota8.zip -O dota8.zip
    !unzip -q -o dota8.zip

# 학습용(train) 4장 + 검증용(val) 4장
IMAGES = sorted(DATA.glob('images/*/*.jpg'))
print(len(IMAGES), '장:', [p.name for p in IMAGES])
im0 = cv2.imread(str(IMAGES[0]))
print('사진 크기(세로, 가로, 색 채널):', im0.shape)

---
## Step 1. 일반 박스 vs 회전 박스

길쭉한 물체(예: 길이 40px, 폭 12px인 차량)가 45° 기울어져 있다고 해 봅시다. 같은 물체를 두 방식으로 감싸고 **박스 넓이 중 물체가 아닌 부분이 얼마나 되는지** 비교합니다.
- 일반 박스(AABB, Axis-Aligned Bounding Box — 화면 가로·세로 축과 나란한 박스): 꼭짓점 2개 `x1, y1, x2, y2`
- 회전 박스(OBB): 중심·크기·각도 `cx, cy, w, h, r` (r은 라디안 단위 회전 각도) 또는 꼭짓점 4개

In [ ]:
# 기울어진 물체 하나를 직접 만들어 두 박스를 비교
L, Wd = 40, 12  # 물체 길이·폭 (픽셀)
canvas = np.full((120, 240, 3), 255, np.uint8)  # 흰 바탕 그림 (세로 120, 가로 240)
rows = []
for k, ang in enumerate([0, 20, 45, 70]):  # 기울기 4가지 (도)
    c = (30 + k * 60, 60)  # 물체 중심 위치
    # cv2.boxPoints: (중심, (폭, 높이), 각도) → 회전 박스 꼭짓점 4개
    obb = cv2.boxPoints((c, (L, Wd), ang)).astype(np.int32)
    x, y, w, h = cv2.boundingRect(obb)  # 회전 박스를 감싸는 일반 박스
    cv2.fillPoly(canvas, [obb], (180, 180, 180))                   # 물체(회색)
    cv2.rectangle(canvas, (x, y), (x + w, y + h), (0, 0, 255), 1)   # 일반 박스(빨강)
    cv2.polylines(canvas, [obb], True, (0, 160, 0), 1)              # 회전 박스(초록)
    rows.append({'angle_deg': ang, 'object_area': L * Wd, 'aabb_area': w * h,
                 'empty_ratio': round(1 - L * Wd / (w * h), 2)})  # 일반 박스 중 빈 부분 비율
show_bgr(cv2.resize(canvas, None, fx=3, fy=3, interpolation=cv2.INTER_NEAREST),
         'red = axis-aligned box, green = rotated box', size=(10, 5))
pd.DataFrame(rows)

기울기가 45°에 가까울수록 일반 박스의 **절반 넘게가 빈 땅**입니다. 물체가 촘촘히 붙어 있으면 이 빈 부분이 이웃 물체를 덮어, 같은 물체를 두 번 찾은 것으로 보고 지워 버리는 문제가 생깁니다(Step 3에서 확인).

---
## Step 2. 사전학습 OBB 모델로 항공 사진 탐지

`yolo26n-obb.pt`는 DOTA 데이터셋(항공·위성 사진에 비행기·선박·차량 등 15종을 회전 박스로 표시한 데이터)으로 미리 학습된 모델입니다. 따로 학습하지 않고 바로 씁니다.
결과의 `r.obb`에는 다음 값이 들어 있습니다.

| 속성 | 모양 | 뜻 |
|---|---|---|
| `xywhr` | N×5 | 중심 x, 중심 y, 폭, 높이, 회전(라디안) |
| `xyxyxyxy` | N×4×2 | 꼭짓점 4개의 픽셀 좌표 |
| `xyxy` | N×4 | 회전 박스를 감싸는 일반 박스 |
| `conf`, `cls` | N | 신뢰도, 클래스 번호 |

In [ ]:
model = YOLO('yolo26n-obb.pt')  # 처음 실행 시 자동 다운로드 (약 6MB)
NAMES = model.names  # 클래스 번호 → 이름 사전
print('클래스', len(NAMES), '개:', list(NAMES.values()))

CONF = 0.25  # 신뢰도(conf — 모델이 매긴 0~1 확신 점수) 기준. 이보다 낮은 예측은 버림
results = model([str(p) for p in IMAGES], conf=CONF, verbose=False)  # 8장을 한 번에 예측

# 예측을 표로 정리: 한 줄 = 회전 박스 하나
rows = []
for p, r in zip(IMAGES, results):
    ob = r.obb
    for (cx, cy, w, h, rad), (x1, y1, x2, y2), cf, c in zip(ob.xywhr.cpu().numpy(), ob.xyxy.cpu().numpy(),
                                                      ob.conf.cpu().numpy(), ob.cls.cpu().numpy()):
        rows.append({'image': p.name, 'cls_name': NAMES[int(c)], 'conf': round(float(cf), 3),
                     'cx': round(float(cx), 1), 'cy': round(float(cy), 1), 'w': round(float(w), 1), 'h': round(float(h), 1),
                     'angle_deg': round(float(np.degrees(rad)), 1),           # 라디안 → 도
                     'obb_area': round(float(w * h)),                          # 회전 박스 넓이
                     'aabb_area': round(float((x2 - x1) * (y2 - y1)))})        # 감싸는 일반 박스 넓이
preds = pd.DataFrame(rows)
preds['area_ratio'] = (preds['aabb_area'] / preds['obb_area']).round(2)  # 일반 박스가 몇 배 더 넓은가
preds.to_csv('obb_preds.csv', index=False); save_out('obb_preds.csv')  # 열 이름은 영문
print('전체 탐지', len(preds), '개')
preds.groupby('cls_name').agg(count=('conf', 'size'), mean_conf=('conf', 'mean'),
                              mean_area_ratio=('area_ratio', 'mean')).round(2).sort_values('count', ascending=False)

In [ ]:
# 탐지가 많은 사진 3장을 골라 결과 그림 보기 (plot(): 회전 박스를 그린 BGR 이미지)
top = preds['image'].value_counts().index[:3]
picked = [(p, r) for p, r in zip(IMAGES, results) if p.name in top]
show_grid([r.plot(line_width=1, font_size=8) for _, r in picked], [p.name for p, _ in picked], cols=3, size=5)

**확인해 보기** — `mean_area_ratio`가 큰 클래스는 무엇인가요? 길쭉하고 비스듬히 놓이는 물체(선박, 대형 차량)일수록 일반 박스가 회전 박스보다 훨씬 넓습니다.

---
## Step 3. 나란히 붙은 물체의 겹침 비교

같은 클래스의 예측 두 개가 얼마나 겹치는지를 **IoU**(Intersection over Union — 겹친 넓이 ÷ 합친 넓이)로 잽니다.
서로 다른 물체라면 겹침이 0이어야 정상입니다. 일반 박스로 바꿔 그리면 이웃한 물체끼리 얼마나 겹치게 되는지 셉니다.
겹침이 커지면 화면에서 물체를 구분하기 어렵고, 더 빽빽해져 IoU가 NMS 기준(보통 0.5~0.7)을 넘으면 YOLO가 "같은 물체를 두 번 찾은 것"으로 보고 하나를 지웁니다(NMS — 겹친 박스 중 신뢰도 높은 것만 남기는 후처리).

In [ ]:
# 다각형 두 개의 IoU: cv2.intersectConvexConvex가 볼록 다각형의 겹친 넓이를 계산
def poly_iou(a, b):
    inter, _ = cv2.intersectConvexConvex(a.astype(np.float32), b.astype(np.float32))
    union = cv2.contourArea(a.astype(np.float32)) + cv2.contourArea(b.astype(np.float32)) - inter
    return inter / union if union > 0 else 0.0

# 일반 박스 [x1, y1, x2, y2]를 꼭짓점 4개 다각형으로 바꿔 같은 함수로 계산
def box_to_poly(b):
    x1, y1, x2, y2 = b
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])

pairs = []
for p, r in zip(IMAGES, results):
    polys = r.obb.xyxyxyxy.cpu().numpy(); boxes = r.obb.xyxy.cpu().numpy(); cls = r.obb.cls.cpu().numpy()
    for i in range(len(cls)):
        for j in range(i + 1, len(cls)):
            if cls[i] != cls[j]:  # 같은 클래스끼리만 비교 (NMS도 클래스별로 함)
                continue
            a = poly_iou(box_to_poly(boxes[i]), box_to_poly(boxes[j]))  # 일반 박스 IoU
            if a == 0:  # 전혀 안 겹치는 쌍은 생략
                continue
            pairs.append({'image': p.name, 'cls_name': NAMES[int(cls[i])],
                          'aabb_iou': round(a, 3), 'obb_iou': round(poly_iou(polys[i], polys[j]), 3)})
pairs = pd.DataFrame(pairs)
print('일반 박스가 조금이라도 겹치는 같은 클래스 쌍:', len(pairs))
for t in [0.1, 0.2]:  # 겹침 기준 두 가지
    print(f'IoU ≥ {t}: 일반 박스 {int((pairs.aabb_iou >= t).sum())}쌍 | 회전 박스 {int((pairs.obb_iou >= t).sum())}쌍')
pairs.sort_values('aabb_iou', ascending=False).head(8)

일반 박스로는 이웃한 배끼리 박스가 겹치는 쌍이 여럿 생기지만, 회전 박스로는 거의 0입니다. 이 사진은 배가 화면 축과 비슷한 방향으로 놓여 있어 겹침이 0.3 아래에 머물지만, 배·차·컨테이너가 **비스듬히 더 빽빽하게** 놓일수록 일반 박스의 겹침이 NMS 기준까지 올라가 실제 물체를 지우게 됩니다. 그런 장면일수록 OBB가 필요합니다.

---
## Step 4. 규칙 — 마리나에서 정박 방향을 벗어난 선박 찾기

"모델이 찾고, 규칙이 판단한다" 구조를 그대로 씁니다. 마리나(요트·보트 정박장)에서 배는 부두(잔교)에 맞춰 같은 방향으로 나란히 묶입니다. 모델은 배의 **위치와 각도**를 주고, 규칙은 다음을 판단합니다.
1. 사진 속 배들의 **주된 방향**(대부분의 배가 놓인 각도)을 구한다.
2. 주된 방향과 `ANGLE_TOL`도 넘게 어긋난 배를 **이동 중이거나 잘못 정박한 배** 로 표시한다.

같은 규칙을 주차장(주차 방향에서 벗어난 차), 컨테이너 야적장(비뚤게 쌓인 컨테이너)에도 쓸 수 있습니다.
각도는 0°와 180°가 같은 방향이므로(배의 앞뒤를 구분하지 않음) 단순 평균을 내면 안 됩니다. 각도를 두 배로 늘려 원 위의 점으로 바꾼 뒤 평균을 내는 **원형 평균**을 씁니다.

In [ ]:
TARGET_CLS = 'ship'  # ✏️ 대상 클래스 ('small vehicle'로 바꾸면 차량에 적용)
ANGLE_TOL = 20             # 주된 방향에서 이 각도(도) 넘게 어긋나면 표시

# 긴 변 방향 각도(0~180도)로 통일: 폭(w)보다 높이(h)가 길면 90도를 더함
def long_axis_deg(w, h, deg):
    return (deg + (90 if h > w else 0)) % 180

# 원형 평균: 각도를 2배 → 단위원 위 점 → 평균 방향 → 다시 ÷2 (0도와 180도를 같은 방향으로)
def dominant_deg(degs):
    t = np.radians(np.array(degs) * 2)
    return (np.degrees(np.arctan2(np.sin(t).mean(), np.cos(t).mean())) / 2) % 180

# 두 방향의 차이 (0~90도)
def angle_diff(a, b):
    d = abs(a - b) % 180
    return min(d, 180 - d)

obj = preds[preds['cls_name'] == TARGET_CLS].copy()
TARGET = obj['image'].value_counts().index[0]  # 대상이 가장 많은 사진
v = obj[obj['image'] == TARGET].copy()
v['axis_deg'] = [long_axis_deg(w, h, a) for w, h, a in zip(v.w, v.h, v.angle_deg)]
DOM = dominant_deg(v['axis_deg'])
v['off_deg'] = [round(angle_diff(a, DOM), 1) for a in v['axis_deg']]
v['flag'] = v['off_deg'] > ANGLE_TOL  # True = 주된 방향에서 벗어남
print(f'{TARGET}: {TARGET_CLS} {len(v)}개 | 주된 방향 {DOM:.1f}도 | 벗어난 것 {int(v.flag.sum())}개')
v[['cls_name', 'conf', 'axis_deg', 'off_deg', 'flag']].sort_values('off_deg', ascending=False).head(10)

In [ ]:
# 결과 그림: 정상 = 초록, 벗어남 = 빨강
PATHS = {p.name: p for p in IMAGES}  # 파일 이름 → 경로
img = cv2.imread(str(PATHS[TARGET]))
r = [r for p, r in zip(IMAGES, results) if p.name == TARGET][0]
polys = r.obb.xyxyxyxy.cpu().numpy().astype(np.int32)
cls = r.obb.cls.cpu().numpy()
k = 0  # v 표의 순서와 맞추기 위한 번호 (같은 사진·같은 클래스 순서대로 들어 있음)
for poly, c in zip(polys, cls):
    if NAMES[int(c)] != TARGET_CLS:
        continue
    bad = bool(v['flag'].iloc[k]); k += 1
    cv2.polylines(img, [poly], True, (0, 0, 255) if bad else (0, 200, 0), 2)
cv2.putText(img, f'dominant {DOM:.0f} deg, flagged {int(v.flag.sum())}/{len(v)}', (15, 35),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
cv2.imwrite('obb_marina.png', img); save_out('obb_marina.png')
show_bgr(img, 'green = aligned, red = off-direction', size=(10, 10))

**생각해 보기** — 빨간 배는 정말 이상한 배일까요? 그림을 보면 빨간 배 대부분은 **비스듬한 잔교에 가지런히 묶인 배**입니다. 사진 전체에 주된 방향 하나를 정한 규칙이 너무 단순했던 것입니다. 실제 관제에 쓰려면 구역(잔교 A / 잔교 B / 항로)을 나눠 **구역마다 주된 방향**을 따로 구하거나, 항로 구역 안의 배는 "이동 중"으로 보는 규칙을 더해야 합니다(2회차 위험구역 실습의 구역 판정과 같은 방식).

---
## Step 5. (선택) 짧은 미세조정

DOTA8은 8장뿐이라 성능을 올리는 학습이 아니라 **OBB 학습이 어떻게 돌아가는지 보는 용도**입니다. 라벨 파일 형식만 확인하고, 시간이 있으면 `TRAIN_DEMO = True`로 바꿔 실행해 보세요.
OBB 라벨은 한 줄에 `클래스 x1 y1 x2 y2 x3 y3 x4 y4`(꼭짓점 4개, 0~1로 나눈 좌표)입니다. 일반 YOLO 라벨(`클래스 cx cy w h`)과 다릅니다.

In [ ]:
lbl = sorted(DATA.glob('labels/train/*.txt'))[0]  # 라벨 파일 하나
print(lbl.name); print(lbl.read_text().splitlines()[0])  # 첫 줄: 클래스 + 꼭짓점 8개 숫자

TRAIN_DEMO = False  # ✏️ True면 10 epoch(에폭 — 학습 데이터 전체를 한 번 보는 단위) 학습 (T4 약 1분)
if TRAIN_DEMO:
    m = YOLO('yolo26n-obb.pt')
    m.train(data='dota8.yaml', epochs=10, imgsz=640, batch=4, project='runs_obb', name='ft', exist_ok=True, verbose=False)
    from IPython.display import Image
    display(Image(filename='runs_obb/ft/results.png', width=900))

---
## 도전 과제

1. **내 사진으로** — 구글 지도·공공 항공사진처럼 위에서 내려다본 주차장·항구 사진을 올려 `model(...)`에 넣어 보세요. 촬영 고도가 DOTA와 많이 다르면 잘 안 찾는 이유도 생각해 보세요.
2. **클래스별 개수 리포트** — `preds`로 사진마다 선박·차량·비행기 수를 세어 "항만 현황표"를 CSV로 만들어 보세요(열 이름은 영문).
3. **워크숍 주제와 연결** — PCB 부품(기울어진 칩), 매장 선반(비스듬한 상품), 문서 속 글자 줄처럼 **방향이 중요한** 대상이 있는 주제라면 OBB 라벨로 바꿔 볼 만합니다.

## 정리

- 회전 박스는 **각도까지** 예측합니다. 비스듬하고 빽빽한 물체에서 빈 공간과 겹침이 크게 줄어듭니다.
- 결과는 `r.obb`(`xywhr`, `xyxyxyxy`)로 꺼냅니다. 일반 탐지의 `r.boxes`와 자리가 다릅니다.
- 각도는 0°와 180°가 같은 방향이므로 **원형 평균**처럼 각도에 맞는 계산을 써야 합니다.
- 모델 출력(위치·각도) 위에 규칙(주된 방향, 허용 각도)을 얹으면 바로 현장 판단이 됩니다.

## 용어 정리

| 용어 | 뜻 |
|---|---|
| OBB (회전 박스) | Oriented Bounding Box. 물체 방향을 따라 기울어진 직사각형입니다. 중심·폭·높이·회전 각도로 나타냅니다. |
| AABB (일반 박스) | Axis-Aligned Bounding Box. 화면의 가로·세로 축과 나란한 직사각형으로, 지금까지 쓴 탐지 박스입니다. |
| xywhr | 회전 박스 표기. 중심 x, 중심 y, 폭, 높이, 회전 각도(라디안)입니다. |
| xyxyxyxy | 회전 박스의 꼭짓점 4개 좌표입니다. OBB 라벨 파일도 이 순서로 적습니다. |
| 라디안 / 도 | 각도 단위입니다. π 라디안 = 180도이며 `np.degrees`로 바꿉니다. |
| DOTA / DOTA8 | 항공·위성 사진 회전 박스 데이터셋입니다. DOTA8은 그중 8장만 담은 맛보기용 샘플입니다. |
| 사전학습 모델 | 큰 데이터로 미리 학습해 둔 모델입니다. `yolo26n-obb.pt`는 DOTA로 학습되어 있습니다. |
| conf (신뢰도) | 모델이 예측마다 매긴 0~1 확신 점수입니다. `CONF`보다 낮은 예측은 버립니다. |
| IoU | 두 영역의 겹친 넓이 ÷ 합친 넓이입니다. 1이면 완전히 같고 0이면 안 겹칩니다. |
| NMS | Non-Maximum Suppression. 크게 겹친 예측 중 신뢰도가 가장 높은 것만 남기는 후처리입니다. |
| 볼록 다각형 | 모든 안쪽 각이 180°보다 작은 다각형입니다. 직사각형은 볼록이라 `cv2.intersectConvexConvex`로 겹친 넓이를 구할 수 있습니다. |
| 원형 평균 | 각도처럼 한 바퀴 돌면 처음으로 돌아오는 값의 평균입니다. 각도를 원 위의 점으로 바꿔 평균 방향을 구합니다. |
| 미세조정 | 사전학습 모델을 내 데이터로 조금 더 학습시키는 것입니다. |
| epoch | 학습 데이터 전체를 한 번 모두 학습하는 단위입니다. |